# 微调技术：如何在 SFT、DPO 和 RFT 之间选择（附 DPO 指南）
 
*本指南面向对 OpenAI API 有一定使用经验、并希望将微调模型用于研究或其他适当用途的开发者和机器学习从业者。OpenAI 的服务不用于任何疾病的个性化治疗或诊断，并受我们的[适用条款](https://openai.com/policies/)约束。*
 
本指南讨论 OpenAI 支持的各种微调方法，重点说明每种方法最适合和不太适合的场景，帮助你为自己的用例找到最合适的技术。随后，本指南将深入介绍其中一种方法——直接偏好优化（Direct Preference Optimization，DPO），并提供其他技术的现有指南链接。
 
**什么是微调？** 微调是指在一个更小的、特定领域的数据集上继续训练，以针对特定任务优化模型的过程。我们通常进行微调主要出于两个原因：
1. 提升模型在特定任务上的表现
2. 提升模型效率（减少所需的 token 数量、将专业能力蒸馏到更小的模型中等）
 
目前，OpenAI 平台支持四种微调方法：
- **监督微调（SFT）：** 这种技术采用传统的监督学习，使用输入-输出对来调整模型参数。训练过程会调整模型权重，以最小化所提供示例中预测输出与目标输出之间的差异。模型将复现它在所提供样本对中发现的特征。
- **视觉微调：** 这种技术将监督微调扩展到多模态数据，在统一的训练框架中同时处理文本和图像。训练过程会调整模型权重，以最小化文本-图像对上的误差，从而提升模型对图像输入的理解能力。
- **直接偏好优化（DPO）：** 这种技术使用成对比较（例如偏好的和被拒绝的示例回复）来优化模型，使其更倾向于某些输出而非其他输出。模型会学习复现所提供比较数据中的偏好模式。
- **强化微调（RFT）：** 这种技术使用带有奖励信号（通过评分器或奖励模型）的强化学习，针对复杂目标微调模型。在 RFT 中，模型在训练期间针对给定提示生成输出，并对每个输出的质量进行评估。然后更新模型参数以最大化奖励，强化那些能带来更好结果的行为。这种迭代式的反馈循环会促使模型改进其推理或决策策略。

为了帮助你选择合适的微调技术，下表总结了每种方法最适合以及不太适合的场景：

| **技术**                            | **适合**                                                                                                                                                                                                                                                                                    | **不适合**                                                                                            |
| ---------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------- |
| **监督微调（SFT）**         | 强化模型中已有的知识。<br>定制回复的结构或语气。<br>生成特定格式的内容。<br>教授复杂指令或纠正指令遵循方面的失败。<br>优化成本/延迟（节省提示中的 token 或进行蒸馏）。 | 添加全新的知识（请考虑改用 RAG）。<br>质量评判具有主观性的任务。                     |
| **视觉微调**                   | 专门的视觉识别任务（例如图像分类）。<br>特定领域的图像理解。<br>纠正复杂提示下指令遵循方面的失败。                                                                               | 纯文本任务。<br>没有特定上下文的通用视觉任务。<br>通用的图像理解。 |
| **直接偏好优化（DPO）** | 使模型输出与主观偏好（语气、礼貌程度）对齐。<br>通过人工评分反馈优化输出。<br>实现细致入微的行为对齐。                                                                                                                                 | 学习全新的任务。<br>没有明确人类偏好信号的任务。                             |
| **强化微调（RFT）**      | 需要高级推理能力的复杂特定领域任务。<br>完善已有的部分能力（促进涌现行为）。<br>具有可衡量反馈的任务。<br>显式标签有限但可以定义奖励信号的场景。                                                                     | 模型完全不具备初始能力的任务。<br>没有明确反馈或可衡量信号的任务。          |




目前，已有以下现成的 Cookbook：

- 监督微调（SFT）：(1) [如何微调聊天模型](https://cookbook.openai.com/examples/how_to_finetune_chat_models) (2) [利用模型蒸馏微调模型](https://cookbook.openai.com/examples/leveraging_model_distillation_to_fine-tune_a_model)
- 视觉微调：[在 GPT-4o 上进行视觉微调以实现视觉问答](https://cookbook.openai.com/examples/multimodal/vision_fine_tuning_on_gpt4o_for_visual_question_answering)
- 强化微调（RFT）：(1) [强化微调（RFT）](https://cookbook.openai.com/examples/reinforcement_fine_tuning)，(2) [针对 healthbench QA 的强化微调](https://cookbook.openai.com/examples/fine-tuned_qa/reinforcement_finetuning_healthbench)

本指南将介绍直接偏好优化（DPO）。

## **直接偏好优化指南**
 
如上所述，[直接偏好优化（DPO）](https://platform.openai.com/docs/guides/direct-preference-optimization) 是一种对齐技术，它使用成对的偏好数据（例如经过排序的回复对）来微调语言模型。DPO 利用显式的成对比较（通常来自人类偏好）直接优化模型，使其更倾向于某些输出而非其他输出。这种方法简化了对齐过程，无需单独的奖励模型或复杂的强化学习流程，使 DPO 成为基于人类反馈的强化学习（RLHF）等技术的轻量级替代方案。
 
什么时候应该使用 DPO？当回复质量具有主观性、无法客观衡量，或者语气、风格、得体性或清晰度等细微标准很重要时——通常是存在多个有效输出的情况——DPO 的表现尤为出色。DPO 在对齐 AI 回复方面特别有效的示例应用包括：
- 增强对话式 AI 的回复
- 提升代码生成的质量与风格
- 确保符合法律、道德与安全标准
- 把控品牌声音、专业度与语气
- 定制创意输出与用户体验

通过在偏好与非偏好补全的显式样本对上进行微调，DPO 使模型输出与这些细微的偏好保持一致。下表给出了一个代表某组织的虚构 AI 助手的成对偏好数据示例，其中偏好的回复清晰、专业，并且符合品牌标准。
 
| **示例问题**                                           | **选中的回复**                                                                                                                                                                   | **被拒绝的回复**                                       |
|------------------------------------------------------|---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|--------------------------------------------------------------------|
| **Q1：** *我该如何评价你们的产品？*              | 要提交产品评价，请访问你的账户仪表盘，选择该产品，然后点击“撰写评价”。分享你的真实体验，为关键功能打分，准备好后即可提交。        | 哟，随便打几颗星就行，没啥大不了的！         |
| **Q2：** *我该如何评价你们的产品？*              | 我们非常欢迎你的反馈！在产品页面的“评价”部分，点击“发表评价”，为产品打分，并添加你喜欢的地方或需要改进之处的评论。                   | 随便写点啥就行——说实话写什么都无所谓。          |
| **Q3：** *如何排查这个特定的错误？* | 要解决错误 ‘X101’，请先清除缓存，然后检查你的网络连接。如果问题仍然存在，请按照 [支持 → 故障排除 → 错误 X101] 中的分步指南操作。 | 重启一下吧，大概。如果还不行，那你就只能靠自己了！ |
 
在本指南中，我们将逐步介绍如何使用微调 API 应用 DPO。你将学习为自己的用例成功运行偏好微调任务所需的关键步骤。
 
我们将涵盖以下内容：
 
- **1. 推荐工作流**
- **2. 演示场景**
- **3. 生成数据集**
- **4. 对基础模型进行基准测试**
- **5. 微调**
- **6. 使用你的微调模型**

## **1. 推荐工作流**
 
OpenAI 推荐以下工作流：
1. 在你偏好的回复的一个子集上执行监督微调（SFT）。
2. 以 SFT 微调后的模型为起点，使用偏好比较数据应用 DPO。
 
在直接偏好优化（DPO）之前执行监督微调（SFT），可以建立一个稳健的初始策略，确保模型已经倾向于正确的回复，从而增强模型对齐效果和整体性能。这减小了 DPO 期间权重更新的幅度，使训练更加稳定，并通过让 DPO 高效地打磨细微差别来防止过拟合。因此，先 SFT 再 DPO 的组合工作流收敛更快，结果质量也更高。

在本指南中，我们将只专注于应用直接偏好优化（DPO）。不过，根据你的用例，先执行监督微调（SFT）可能会带来性能提升。如果是这样，你可以按照上面链接的 SFT 指南操作，保存得到的模型 ID，并将其作为 DPO 任务的起点。

## **2. 演示场景**

为了更具体，我们来逐步演示如何微调一个面向客户的 AI 助手，使其遵循一个虚构品牌的声音和风格。设想一家名为 Good Vibes Corp 的组织，它以友好、热情且富有人情味的语气为傲。
 
他们希望自己的客户 AI 助手在回答问题时能体现这些品牌准则（例如积极向上的态度、礼貌的用语和友好的结束语），并且更偏好这类回复，而不是更笼统或生硬的回答。这正是 DPO 的用武之地：不存在客观正确的回答格式，但存在一种偏好的风格。
 
DPO 将帮助模型通过比较来学习哪种风格更受偏好。我们将概述以下步骤：(1) 生成一个合成偏好数据集，其中每个提示都配有成对的回复（一个符合期望的品牌声音，一个不符合）。(2) 使用 OpenAI evals API 评估基础模型的表现。(3) 按照偏好微调所需的 JSONL 格式准备并上传数据。(4) 使用 OpenAI 微调 API 通过 DPO 微调模型。(5) 使用 OpenAI evals API 评估微调后的模型，展示其在品牌风格偏好方面的改进。

我们将为本演示合成一个数据集。首先，我们创建一个种子问题库，用于从中生成更多变体。

让我们开始吧！

In [1]:
! pip install openai nest-asyncio --quiet

In [1]:
PROMPT_SEED_POOL = [
    "Hi, I ordered a gadget last week. When will it arrive?",
    "Your product stopped working after two days. Can I get help?",
    "Do you offer discounts for long-term customers?",
    "Can I change the shipping address for my order?",
    "What is your return policy for damaged items?",
    "My tracking number hasn't updated in three days—can you check the status?",
    "How long is the warranty on your products, and how do I submit a claim?",
    "Can I add gift wrapping to my order before it ships?",
    "Do you accept PayPal or other alternative payment methods?",
    "Is there an option to expedite shipping if my order hasn't left the warehouse yet?",
]

## **3. 生成数据集**

接下来，我们将定义一些函数，从种子库中取出每个提示并生成相关问题。我们会先生成这些提示变体，然后为每个提示同时生成一个偏好的回复和一个被拒绝的回复，从而创建一个偏好对数据集。

该数据集是合成的，用于说明直接偏好优化的机制——在开发你自己的应用时，你应该收集或整理一个高质量的偏好数据集。注意：DPO 所需的数据量取决于用例；一般来说越多越好（数千到数万条），并且对于偏好对，排序逻辑应保持一致（例如，如果 A > B 且 B > C，则 A > C）。

In [2]:
import asyncio
from openai import AsyncOpenAI
from typing import List, Dict, Any

async_client = AsyncOpenAI()

SYSTEM_PROMPT = "You are a customer-support assistant."


async def _generate_related_questions_from_prompt(
    prompt: str, k: int, sem: asyncio.Semaphore, *, model: str
) -> List[str]:
    """Return *k* distinct customer-service questions related to the given prompt."""
    out: List[str] = []
    async with sem:
        for _ in range(k):
            resp = await async_client.responses.create(
                model=model,
                input=[
                    {
                        "role": "system",
                        "content": (
                            "Return ONE distinct, realistic customer-service question "
                            "related in topic or theme to the following question, "
                            "but NOT a direct paraphrase."
                        ),
                    },
                    {"role": "user", "content": prompt},
                ],
                temperature=0.9,
                max_output_tokens=60,
            )
            out.append(resp.output_text.strip())
    return out


async def expand_prompt_pool(
    prompts: List[str], *, k: int = 3, concurrency: int = 32, model: str
) -> List[str]:
    """Expand each prompt into *k* related questions using the given model."""
    sem = asyncio.Semaphore(concurrency)
    tasks = [
        _generate_related_questions_from_prompt(p, k, sem, model=model) for p in prompts
    ]
    results = await asyncio.gather(*tasks)
    return [v for sub in results for v in sub]


async def _generate_preference_pair(
    prompt: str, sem: asyncio.Semaphore, *, model: str
) -> Dict[str, Any]:
    """Generate a preference pair for the given prompt."""
    async with sem:
        friendly_task = async_client.responses.create(
            model=model,
            input=[
                {
                    "role": "system",
                    "content": (
                        "You are Good Vibes Corp's exceptionally energetic, outrageously friendly and "
                        "enthusiastic support agent."
                    ),
                },
                {"role": "user", "content": prompt},
            ],
            temperature=0.7,  # higher temperature to increase creativity & on-brand tone adherence
            max_output_tokens=80,
        )
        blunt_task = async_client.responses.create(
            model=model,
            input=[
                {
                    "role": "system",
                    "content": "You are a terse, factual support agent with no empathy or politeness.",
                },
                {"role": "user", "content": prompt},
            ],
            temperature=0.3,  # lower temperature to limit creativity & emphasize tonal difference
            max_output_tokens=80,
        )
        friendly, blunt = await asyncio.gather(friendly_task, blunt_task)
        return {
            "input": {
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt},
                ]
            },
            "preferred_output": [
                {"role": "assistant", "content": friendly.output_text}
            ],
            "non_preferred_output": [
                {"role": "assistant", "content": blunt.output_text}
            ],
        }

现在，我们将使用这些定义好的函数，通过生成友好与生硬的回复对来构建数据集。友好的回复体现了品牌期望的沟通风格。为了提高效率，我们将以异步方式完成这项工作，创建一个适用于直接偏好优化的数据集。

In [3]:
import math
import nest_asyncio


async def build_dataset(
    *,
    pair_count: int = 500,
    concurrency: int = 8,
    expand_prompt_pool_model: str,
    generate_preference_pair_model: str,
) -> List[Dict[str, Any]]:
    """Return *pair_count* preference pairs (single-shot expansion)."""

    seed = PROMPT_SEED_POOL
    deficit = max(0, pair_count - len(seed))
    k = max(1, math.ceil(deficit / len(seed)))

    expanded = await expand_prompt_pool(
        seed,
        k=k,
        concurrency=concurrency,
        model=expand_prompt_pool_model,
    )
    prompt_bank = (seed + expanded)[:pair_count]

    sem = asyncio.Semaphore(concurrency)
    tasks = [
        _generate_preference_pair(p, sem, model=generate_preference_pair_model)
        for p in prompt_bank
    ]
    return await asyncio.gather(*tasks)


nest_asyncio.apply()
pairs = await build_dataset(
    pair_count=500,
    concurrency=8,
    expand_prompt_pool_model="gpt-4.1-mini-2025-04-14",
    generate_preference_pair_model="gpt-4.1-mini-2025-04-14",
)
print(f"Dataset ready with {len(pairs)} pairs.")

Dataset ready with 500 pairs.


## **4. 对基础模型进行基准测试**

下面，我们将数据集划分为训练集、验证集和测试集。我们还展示了训练数据集中的一个样本，它清楚地体现了该输入对中偏好的回复（友好、符合品牌）与非偏好的回复（生硬、中性）之间的差异。

In [4]:
# set dataset sizes
n = len(pairs)
n_train = int(0.8 * n)
n_val = int(0.1 * n)
n_test = n - n_train - n_val

# split dataset into train, test & validation
train_pairs = pairs[:n_train]
val_pairs = pairs[n_train : n_train + n_val]
test_pairs = pairs[n_train + n_val :]
train_pairs[0]

{'input': {'messages': [{'role': 'system',
    'content': 'You are a customer-support assistant.'},
   {'role': 'user',
    'content': 'Hi, I ordered a gadget last week. When will it arrive?'}]},
 'preferred_output': [{'role': 'assistant',
   'content': 'Hey there, awesome friend! 🌟 Thanks a bunch for reaching out! I’d LOVE to help you track down your gadget so you can start enjoying it ASAP! 🎉 Could you please share your order number or the email you used to place the order? Let’s make this delivery magic happen! 🚀✨'}],
 'non_preferred_output': [{'role': 'assistant',
   'content': 'Provide your order number for delivery status.'}]}

为了评估模型在微调前的表现，我们将使用一个自动评分器（LLM-as-a-Judge）对每个回复的友好度和同理心进行打分。评分器会为每个回答给出 0 到 4 分，让我们可以计算出基础模型的平均基线分数。

为此，我们首先在测试集上为基础模型生成回复，然后使用 OpenAI evals API 创建并运行一个带有自动评分器的评估。

In [5]:
async def generate_responses(
    testset,
    model,
    temperature=0.0,
    max_output_tokens=80,
    concurrency=8,
):
    """
    Generate responses for each prompt in the testset using the OpenAI responses API.
    Returns: List of dicts: [{"prompt": ..., "response": ...}, ...]
    """
    async_client = AsyncOpenAI()
    sem = asyncio.Semaphore(concurrency)

    async def get_response(prompt):
        async with sem:
            resp = await async_client.responses.create(
                model=model,
                input=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt},
                ],
                temperature=temperature,
                max_output_tokens=max_output_tokens,
            )
            return {"prompt": prompt, "response": resp.output_text}

    tasks = [get_response(item["item"]["input"]) for item in testset]
    results = await asyncio.gather(*tasks)
    return results


# generate responses for the base model over the test set
base_model = "gpt-4.1-mini-2025-04-14"
testset = [
    {"item": {"input": pair["input"]["messages"][1]["content"]}} for pair in test_pairs
]
responses = await generate_responses(testset, model=base_model)

接下来，我们将使用 OpenAI evals API 创建并运行一个带有自动评分器的评估，首先定义 LLM-as-a-Judge 的评分标准。注意：我们将通过数据日志访问回复，因此要使其正常工作，你所在的组织需要未禁用数据日志（例如通过 zdr 等方式禁用）。如果你不确定自己是否属于这种情况，请前往 https://platform.openai.com/logs?api=responses 查看是否能看到刚刚生成的回复。

In [6]:
JUDGE_SYSTEM = """
You judge whether a reply matches Good Vibes Corp's desired tone:
energetic, super-friendly, enthusiastic.

Score 0-4 (higher = more energy):

4 - Highly enthusiastic: multiple upbeat phrases / emojis / exclamations, clear empathy, proactive help.
3 - Energetic & friendly: visible enthusiasm cue (≥1 emoji OR exclamation OR upbeat phrase), warm second-person tone.
2 - Pleasant: polite & positive but lacks obvious enthusiasm cues.
1 - Neutral: correct, businesslike, minimal warmth.
0 - Rude, negative, or unhelpful.
"""

In [7]:
from openai import OpenAI

sync_client = OpenAI()

# set judge model
judge_model = "gpt-4.1-2025-04-14"

# create the evaluation
logs_eval = sync_client.evals.create(
    name="Good Vibes Corp Tone Eval",
    data_source_config={
        "type": "logs",
    },
    testing_criteria=[
        {
            "type": "score_model",
            "name": "General Evaluator",
            "model": judge_model,
            "input": [
                {
                    "role": "system",
                    "content": JUDGE_SYSTEM,
                },
                {
                    "role": "user",
                    "content": (
                        "**User input**\n"
                        "{{item.input}}\n"
                        "**Response to evaluate**\n"
                        "{{sample.output_text}}"
                    ),
                },
            ],
            "range": [0, 4],
            "pass_threshold": 2,
        }
    ],
)

In [8]:
# run the evaluation
base_run = sync_client.evals.runs.create(
    name=base_model,
    eval_id=logs_eval.id,
    data_source={
        "type": "responses",
        "source": {"type": "responses", "limit": len(test_pairs)},
    },
)

In [10]:
# score base model
base_data = sync_client.evals.runs.output_items.list(
    eval_id=logs_eval.id, run_id=base_run.id
).data
base_scores = [s.results[0]["score"] for s in base_data]
print("Average score:", sum(base_scores) / len(base_scores))

Average score: 2.525


## **5. 微调**

建立基线后，我们现在可以使用训练集和 DPO 对模型进行微调。这一过程将基于我们之前创建的偏好对，教会模型更倾向于符合我们期望风格的回复。

注意：**beta (β)** 是直接偏好优化（DPO）特有的微调超参数。它是一个介于 0 到 2 之间的浮点数，用于控制保留模型现有行为与适应新的、符合偏好的回复之间的平衡。
- 高 β（接近 2）：使模型更加保守，强烈倾向于之前的行为。微调后的模型与其原始风格或特征的偏离会很小，强调一致性并避免突变。
- 中等 β（约 1）：在遵循先前行为与适应新偏好之间取得平衡。推荐作为大多数实际场景的合理起点。
- 低 β（接近 0）：鼓励激进的适应，使模型更加突出地优先考虑新提供的偏好。这可能会带来显著的风格转变以及与显式偏好更高的一致性，但也可能导致意料之外或过度专门化的输出。

从技术上讲，beta 会缩放 DPO 损失中对数概率的差值；较大的 β 会使基于 sigmoid 的损失函数在较小的概率差下就趋于饱和，从而产生较小的权重更新（因此保留旧的行为）。建议系统地试验 β 值，以针对你的具体用例以及在稳定性与适应性之间期望的权衡取得最佳结果。

In [11]:
import io
import json

# create training file
train_buf = io.BytesIO("\n".join(json.dumps(p) for p in train_pairs).encode())
train_buf.name = "train.jsonl"
train_file_id = sync_client.files.create(file=train_buf, purpose="fine-tune").id

# create validation file
val_buf = io.BytesIO("\n".join(json.dumps(p) for p in val_pairs).encode())
val_buf.name = "val.jsonl"
val_file_id = sync_client.files.create(file=val_buf, purpose="fine-tune").id

# create a fine-tuning job
ft = sync_client.fine_tuning.jobs.create(
    model=base_model,
    training_file=train_file_id,
    validation_file=val_file_id,
    method={
        "type": "dpo",
        "dpo": {
            "hyperparameters": {
                "n_epochs": 2,
                "beta": 0.1,
                "batch_size": 8,
            }
        },
    },
)
print(f"Fine-tuning job created: job_id = {ft.id}")

Fine-tuning job created: job_id = ftjob-5QPmA36QezFRGoXjuvIAPuAQ


## **6. 使用你的微调模型**

微调完成后，我们将在同一测试集上评估经过 DPO 调优的模型。通过比较微调前后的平均分数，并审阅示例输出，我们可以看到模型与我们偏好的对齐程度提升了多少。

In [ ]:
# generate responses
job = sync_client.fine_tuning.jobs.retrieve(ft.id)
if job.status == "succeeded":
    responses = await generate_responses(testset, model=job.fine_tuned_model)

    post_run = sync_client.evals.runs.create(
        name=ft.id,
        eval_id=logs_eval.id,
        data_source={
            "type": "responses",
            "source": {"type": "responses", "limit": len(test_pairs)},
        },
    )

In [59]:
# get scores from the evaluation
post_data = sync_client.evals.runs.output_items.list(
    eval_id=logs_eval.id, run_id=post_run.id
).data
post_scores = [s.results[0]["score"] for s in post_data]

# print scores & a sample comparison from the test set for illustration
print(
    "Δ mean:",
    sum(t - b for b, t in zip(base_scores, post_scores)) / len(base_scores),
)
print("\n=== SAMPLE COMPARISON ===")
idx = 0
print(f"Prompt:\n  {testset[idx]['item']['input']}\n")
print(f"Base model reply: \n {base_data[idx].sample.output[0].content} \n")
print(f"DPO-tuned model reply \n {post_data[idx].sample.output[0].content}")

Δ mean: 0.45

=== SAMPLE COMPARISON ===
Prompt:
  Can I upgrade to faster delivery if my package is still being processed?

Base model reply: 
 Whether you can upgrade to express shipping while your order is still being processed depends on the store's policies. Generally, many stores allow shipping upgrades before the order is shipped. 

To assist you better, could you please provide your order number or the name of the store you ordered from? Alternatively, you can contact the store's customer service directly to request the upgrade. 

DPO-tuned model reply 
 Hi! I’d be happy to help with that. If your package hasn’t shipped yet, there’s a good chance we can upgrade your delivery speed. Could you please provide me with your order number? I’ll check the status and let you know the available options for faster delivery.
